# DS1/GSE103224 — dataset-level quality control

This notebook applies the dataset-specific QC rules reported for
DS1/GSE103224.

## Filtering sequence

1. calculate cell-level QC metrics from raw integer counts;
2. apply sample-specific adaptive upper thresholds:
   - detected genes: 99.5th percentile;
   - total counts: 99.5th percentile;
   - mitochondrial fraction: the lower of 20% and the sample-specific
     99th percentile;
3. require at least 300 detected genes and 500 total counts;
4. initially remove genes detected in fewer than 3 retained cells;
5. apply the final total-count ceiling of 15,000;
6. retain genes detected in at least 10 final cells.

Raw counts remain in both `adata.X` and `adata.layers["counts"]`.
No normalization or log transformation is performed.

## Expected output

- 23,207 cells
- 29,457 genes

Input:
`data/interim/DS1_GSE103224/GSE103224_all_samples_inner.h5ad`

Output:
`data/processed/DS1_GSE103224/GSE103224_postQC.h5ad`

> **AnnData 0.12 compatibility patch:** backed `AnnData` objects are
> explicitly closed with `check.file.close()` rather than used as
> context managers.

In [1]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)

Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
scanpy: 1.11.5
pandas: 2.3.3


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_22768\3006837562.py:14: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


In [2]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE103224_INTERIM_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS1_GSE103224"
        / "GSE103224_all_samples_inner.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE103224_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS1_GSE103224"
        / "GSE103224_postQC.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT / "results" / "qc" / "DS1_GSE103224" / "00b_qc"
)
FIGURE_DIR = AUDIT_DIR / "figures"

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_INPUT_SHAPE = (23_793, 60_725)
EXPECTED_OUTPUT_SHAPE = (23_207, 29_457)

MIN_GENES = 300
MIN_COUNTS = 500
MAX_GENES_QUANTILE = 0.995
MAX_COUNTS_QUANTILE = 0.995
MAX_MT_QUANTILE = 0.99
MAX_MT_CAP = 20.0
INITIAL_MIN_CELLS_PER_GENE = 3
FINAL_MAX_COUNTS = 15_000
FINAL_MIN_CELLS_PER_GENE = 10

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Input AnnData object not found: {INPUT_H5AD}\n"
        "Run 00a_ds1_build_raw_anndata.ipynb first or set "
        "GSE103224_INTERIM_H5AD."
    )

Input: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\interim\DS1_GSE103224\GSE103224_all_samples_inner.h5ad
Output: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS1_GSE103224\GSE103224_postQC.h5ad


In [3]:
def sparse_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_nonnegative_integer_counts(matrix, label: str):
    values = sparse_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} has no nonzero count values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")


def save_current_figure(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close("all")

In [4]:
adata = sc.read_h5ad(INPUT_H5AD)

if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected input shape {EXPECTED_INPUT_SHAPE}, observed {adata.shape}"
    )
if "sample_id" not in adata.obs:
    raise KeyError("sample_id is missing from adata.obs.")
if "gene_symbol" not in adata.var:
    raise KeyError("gene_symbol is missing from adata.var.")
if not adata.obs_names.is_unique:
    raise AssertionError("Cell identifiers are not unique.")
if not adata.var_names.is_unique:
    raise AssertionError("Feature identifiers are not unique.")

assert_raw_nonnegative_integer_counts(adata.X, "adata.X")

# Preserve a dedicated count layer without creating a full .raw duplicate.
adata.layers["counts"] = adata.X.copy()
assert_raw_nonnegative_integer_counts(
    adata.layers["counts"], 'adata.layers["counts"]'
)

print(adata)

AnnData object with n_obs × n_vars = 23793 × 60725
    obs: 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset'
    var: 'ensembl_id_version', 'gene_symbol'
    uns: 'build_timestamp', 'dataset_id', 'geo_accession', 'source_files'
    layers: 'counts'


In [7]:
gene_symbols = (
    adata.var["gene_symbol"]
    .astype("string")
    .fillna("")
    .str.upper()
)

adata.var["mt"] = (
    gene_symbols.str.startswith("MT-", na=False)
    .to_numpy(dtype=bool)
)

adata.var["ribo"] = (
    gene_symbols.str.startswith(("RPS", "RPL"), na=False)
    .to_numpy(dtype=bool)
)

adata.var["hb"] = (
    gene_symbols.str.match(r"^HB[ABDEGQMZ]", na=False)
    .to_numpy(dtype=bool)
)

# Fail early if pandas converts any mask back to an extension dtype.
for qc_var in ("mt", "ribo", "hb"):
    assert adata.var[qc_var].dtype == bool, (
        f"{qc_var} has unexpected dtype: {adata.var[qc_var].dtype}"
    )

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    percent_top=[50, 100, 200, 500],
    inplace=True,
    log1p=True,
)

In [9]:
import numpy as np
import pandas as pd


# Ensure that all QC columns are ordinary numeric columns rather than
# pandas categorical or nullable extension arrays.
qc_numeric_columns = [
    "n_genes_by_counts",
    "total_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "pct_counts_hb",
]

missing_qc_columns = [
    column
    for column in qc_numeric_columns
    if column not in adata.obs.columns
]
assert not missing_qc_columns, (
    f"Missing QC columns: {missing_qc_columns}"
)

for column in qc_numeric_columns:
    adata.obs[column] = pd.to_numeric(
        adata.obs[column].astype("string"),
        errors="raise",
    ).astype("float64")

for column in qc_numeric_columns:
    assert pd.api.types.is_numeric_dtype(adata.obs[column]), (
        f"{column} is not numeric: {adata.obs[column].dtype}"
    )
    assert adata.obs[column].notna().all(), (
        f"{column} contains missing values"
    )


# Calculate adaptive thresholds independently for each sample.
sample_groups = adata.obs.groupby(
    "sample_id",
    observed=True,
)

thresholds = pd.DataFrame(
    {
        "max_genes_q995": sample_groups[
            "n_genes_by_counts"
        ].quantile(MAX_GENES_QUANTILE),

        "max_counts_q995": sample_groups[
            "total_counts"
        ].quantile(MAX_COUNTS_QUANTILE),

        "max_mt_q99": sample_groups[
            "pct_counts_mt"
        ].quantile(MAX_MT_QUANTILE),
    }
).astype("float64")

thresholds["max_mt_applied"] = np.minimum(
    float(MAX_MT_CAP),
    thresholds["max_mt_q99"].to_numpy(dtype=np.float64),
)

thresholds.index.name = "sample_id"

assert thresholds.index.is_unique, (
    "Threshold table contains duplicated sample IDs"
)
assert thresholds.notna().all().all(), (
    "Threshold table contains missing values"
)

thresholds.to_csv(
    AUDIT_DIR / "GSE103224_adaptive_QC_thresholds_by_sample.tsv",
    sep="\t",
)


# Expand sample-level thresholds to cell-level numeric arrays.
sample_ids = adata.obs["sample_id"].astype("string")

assert sample_ids.notna().all(), (
    "adata.obs['sample_id'] contains missing values"
)

threshold_lookup = thresholds.copy()
threshold_lookup.index = threshold_lookup.index.astype("string")

sample_id_values = sample_ids.to_numpy(dtype=str)


def expand_sample_threshold(column):
    expanded = threshold_lookup[column].reindex(sample_id_values)

    assert expanded.notna().all(), (
        f"Missing mapped values for threshold: {column}"
    )

    return expanded.to_numpy(dtype=np.float64)


upper_genes = expand_sample_threshold("max_genes_q995")
upper_counts = expand_sample_threshold("max_counts_q995")
upper_mt = expand_sample_threshold("max_mt_applied")


# Convert QC measurements to ordinary NumPy arrays to avoid categorical
# or pandas BooleanArray comparisons.
n_genes = adata.obs["n_genes_by_counts"].to_numpy(dtype=np.float64)
total_counts = adata.obs["total_counts"].to_numpy(dtype=np.float64)
pct_counts_mt = adata.obs["pct_counts_mt"].to_numpy(dtype=np.float64)


# Apply individual QC rules.
rule_masks = pd.DataFrame(index=adata.obs_names)

rule_masks["pass_min_genes"] = (
    n_genes >= float(MIN_GENES)
)

rule_masks["pass_min_counts"] = (
    total_counts >= float(MIN_COUNTS)
)

rule_masks["pass_max_genes_adaptive"] = (
    n_genes <= upper_genes
)

rule_masks["pass_max_counts_adaptive"] = (
    total_counts <= upper_counts
)

rule_masks["pass_max_mt_adaptive"] = (
    pct_counts_mt <= upper_mt
)

rule_masks["pass_adaptive_QC"] = rule_masks.all(axis=1)


# Confirm that every rule is stored as an ordinary boolean column.
for column in rule_masks.columns:
    assert rule_masks[column].dtype == bool, (
        f"{column} has unexpected dtype: "
        f"{rule_masks[column].dtype}"
    )


# Save the number of cells passing and failing each QC rule.
rule_summary = pd.DataFrame(
    {
        "rule": rule_masks.columns,
        "cells_passing": [
            int(rule_masks[column].sum())
            for column in rule_masks.columns
        ],
        "cells_failing": [
            int((~rule_masks[column]).sum())
            for column in rule_masks.columns
        ],
    }
)

rule_summary.to_csv(
    AUDIT_DIR / "GSE103224_QC_rule_summary.tsv",
    sep="\t",
    index=False,
)

thresholds

,max_genes_q995,max_counts_q995,max_mt_q99,max_mt_applied
sample_id,,,,
PJ016,6425.320,22950.50,4.328085,4.328085
PJ017,4337.100,13068.30,4.540832,4.540832
PJ018,4518.320,11074.52,3.060175,3.060175
PJ025,3857.155,10220.08,6.894315,6.894315
PJ030,2898.320,5582.48,2.044297,2.044297
PJ032,4173.200,10791.32,1.660750,1.660750
PJ035,2304.155,3875.66,1.947160,1.947160
PJ048,3552.005,8010.87,0.686125,0.686125


In [10]:
before_sample_counts = (
    adata.obs.groupby("sample_id", observed=True)
    .size()
    .rename("cells_before_QC")
)

adata_qc = adata[rule_masks["pass_adaptive_QC"].to_numpy()].copy()

sc.pp.filter_genes(
    adata_qc,
    min_cells=INITIAL_MIN_CELLS_PER_GENE,
)

# Final count ceiling reported for DS1.
adata_qc = adata_qc[
    adata_qc.obs["total_counts"].to_numpy() <= FINAL_MAX_COUNTS
].copy()

sc.pp.filter_genes(
    adata_qc,
    min_cells=FINAL_MIN_CELLS_PER_GENE,
)

after_sample_counts = (
    adata_qc.obs.groupby("sample_id", observed=True)
    .size()
    .rename("cells_after_QC")
)

sample_retention = pd.concat(
    [before_sample_counts, after_sample_counts],
    axis=1,
).fillna(0)
sample_retention["cells_removed"] = (
    sample_retention["cells_before_QC"]
    - sample_retention["cells_after_QC"]
)
sample_retention["retention_fraction"] = (
    sample_retention["cells_after_QC"]
    / sample_retention["cells_before_QC"]
)
sample_retention.to_csv(
    AUDIT_DIR / "GSE103224_QC_retention_by_sample.tsv",
    sep="\t",
)

if adata_qc.shape != EXPECTED_OUTPUT_SHAPE:
    raise AssertionError(
        "DS1 QC did not reproduce the reported output shape. "
        f"Expected {EXPECTED_OUTPUT_SHAPE}, observed {adata_qc.shape}. "
        "Confirm that the exact GEO files and compatible package versions "
        "are being used."
    )

assert_raw_nonnegative_integer_counts(adata_qc.X, "post-QC adata.X")
assert_raw_nonnegative_integer_counts(
    adata_qc.layers["counts"],
    'post-QC adata.layers["counts"]',
)

print("Post-QC object:", adata_qc)
sample_retention

Post-QC object: AnnData object with n_obs × n_vars = 23207 × 29457
    obs: 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb'
    var: 'ensembl_id_version', 'gene_symbol', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    uns: 'build_timestamp', 'dataset_id', 'geo_accession', 'source_files'
    layers: 'counts'


,cells_before_QC,cells_after_QC,cells_removed,retention_fraction
sample_id,,,,
PJ016,3085,2838,247,0.919935
PJ017,1261,1237,24,0.980967
PJ018,2197,2163,34,0.984524
PJ025,5924,5829,95,0.983964
PJ030,3097,3048,49,0.984178
PJ032,1377,1354,23,0.983297
PJ035,3768,3707,61,0.983811
PJ048,3084,3031,53,0.982815


In [11]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE103224_QC_metrics_before_filtering.png")

sc.pl.violin(
    adata_qc,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE103224_QC_metrics_after_filtering.png")

sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE103224_scatter_before_filtering.png")

sc.pl.scatter(
    adata_qc,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE103224_scatter_after_filtering.png")

In [12]:
qc_summary = {
    "dataset_id": "DS1_GSE103224",
    "geo_accession": "GSE103224",
    "input_cells": int(adata.n_obs),
    "input_genes": int(adata.n_vars),
    "post_QC_cells": int(adata_qc.n_obs),
    "post_QC_genes": int(adata_qc.n_vars),
    "retention_fraction": float(adata_qc.n_obs / adata.n_obs),
    "min_genes": MIN_GENES,
    "min_counts": MIN_COUNTS,
    "max_genes_quantile_within_sample": MAX_GENES_QUANTILE,
    "max_counts_quantile_within_sample": MAX_COUNTS_QUANTILE,
    "max_mt_quantile_within_sample": MAX_MT_QUANTILE,
    "max_mt_cap_percent": MAX_MT_CAP,
    "initial_min_cells_per_gene": INITIAL_MIN_CELLS_PER_GENE,
    "final_max_counts": FINAL_MAX_COUNTS,
    "final_min_cells_per_gene": FINAL_MIN_CELLS_PER_GENE,
    "raw_counts_in_X": True,
    "raw_counts_layer": "counts",
    "normalization_applied": False,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
}

adata_qc.uns["dataset_level_QC"] = qc_summary

with open(
    AUDIT_DIR / "GSE103224_QC_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(qc_summary, handle, indent=2)

print(qc_summary)

{'dataset_id': 'DS1_GSE103224', 'geo_accession': 'GSE103224', 'input_cells': 23793, 'input_genes': 60725, 'post_QC_cells': 23207, 'post_QC_genes': 29457, 'retention_fraction': 0.9753709074097424, 'min_genes': 300, 'min_counts': 500, 'max_genes_quantile_within_sample': 0.995, 'max_counts_quantile_within_sample': 0.995, 'max_mt_quantile_within_sample': 0.99, 'max_mt_cap_percent': 20.0, 'initial_min_cells_per_gene': 3, 'final_max_counts': 15000, 'final_min_cells_per_gene': 10, 'raw_counts_in_X': True, 'raw_counts_layer': 'counts', 'normalization_applied': False, 'timestamp': '2026-08-03T15:05:26', 'output_h5ad': 'C:\\Users\\AmirSilco\\Documents\\GitHub\\glioma-cnv-single-cell-atlas\\data\\processed\\DS1_GSE103224\\GSE103224_postQC.h5ad'}


In [13]:
temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata_qc.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

check = sc.read_h5ad(OUTPUT_H5AD, backed="r")
try:
    if check.shape != EXPECTED_OUTPUT_SHAPE:
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )
    if "counts" not in check.layers:
        raise AssertionError("Saved output is missing layers['counts'].")
finally:
    check.file.close()

print("Saved and validated:", OUTPUT_H5AD)

Saved and validated: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS1_GSE103224\GSE103224_postQC.h5ad
